# TFM - Predicción Horaria del Consumo HVAC a 24 horas

Este notebook organiza el experimento `Direct-24` como una historia analítica completa: desde el problema de negocio hasta el diagnóstico honesto del desempeño.

**Caso de uso**
- El objetivo no es predecir un único punto aislado, sino anticipar el consumo HVAC de las próximas 24 horas para apoyar planificación operativa.
- Una buena solución de negocio no solo debe reducir error medio; también debe capturar el patrón diario, responder razonablemente al arranque del edificio y ofrecer una señal suficientemente estable para apoyar decisiones.

**Idea central del notebook**
    - Trabajar con datos remuestreados a frecuencia horaria.
    - Construir un problema `Direct-24`, donde cada origen temporal predice el bloque completo `t+1 ... t+24`.
    - Comparar varias versiones de XGBoost y, después, probar redes neuronales secuenciales para verificar si un modelo más complejo rompe el techo observado.
    


In [ ]:
    import random
    import warnings
    from pathlib import Path

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import seaborn as sns
    from sklearn.metrics import r2_score
    from sklearn.preprocessing import RobustScaler

    try:
        import xgboost as xgb
        XGB_IMPORT_ERROR = None
    except Exception as exc:
        xgb = None
        XGB_IMPORT_ERROR = exc

    try:
        import torch
        import torch.nn as nn
        from torch.utils.data import Dataset, DataLoader
        TORCH_IMPORT_ERROR = None
    except Exception as exc:
        torch = None
        nn = None
        Dataset = None
        DataLoader = None
        TORCH_IMPORT_ERROR = exc

    if xgb is None:
        raise RuntimeError(f"XGBoost no esta disponible en este entorno: {XGB_IMPORT_ERROR}")

    warnings.filterwarnings("ignore")
    pd.set_option("display.float_format", "{:.4f}".format)
    sns.set_theme(style="whitegrid")
    plt.rcParams["figure.dpi"] = 130

    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)

    if torch is not None:
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        DEVICE = None

    BASE_DIR = Path.cwd()

    def _find_existing(path_candidates, filename=None, search_roots=None):
        """Busca un archivo existente en varias ubicaciones posibles."""
        candidates = []
        for raw in path_candidates or []:
            if raw is None:
                continue
            path = Path(raw).expanduser()
            if not path.is_absolute():
                path = (BASE_DIR / path).resolve()
            else:
                path = path.resolve()
            candidates.append(path)

        for path in candidates:
            if path.exists():
                return path

        if search_roots is None:
            search_roots = [BASE_DIR, BASE_DIR.parent]

        for root in search_roots:
            root = Path(root).expanduser()
            if not root.is_absolute():
                root = (BASE_DIR / root).resolve()
            else:
                root = root.resolve()
            if not root.exists():
                continue
            if filename is None:
                continue
            matches = [p for p in root.rglob(filename) if p.is_file()]
            if matches:
                return sorted(matches, key=lambda p: len(str(p)))[0]

        return None

    DATA_PATH = _find_existing(
        [
            Path(__import__("os").environ.get("E3LAB_DATASET_PATH", "")).expanduser() if __import__("os").environ.get("E3LAB_DATASET_PATH") else None,
            "E3Lab_modelo_1h.csv",
            "data/E3Lab_modelo_1h.csv",
            "datasets/E3Lab_modelo_1h.csv",
            BASE_DIR.parent / "E3Lab_modelo_1h.csv",
            BASE_DIR.parent / "data" / "E3Lab_modelo_1h.csv",
            BASE_DIR.parent / "datasets" / "E3Lab_modelo_1h.csv",
        ],
        filename="E3Lab_modelo_1h.csv",
        search_roots=[BASE_DIR, BASE_DIR.parent],
    )
    FEATURE_SETS_PATH = _find_existing(
        [
            Path(__import__("os").environ.get("E3LAB_FEATURE_SETS_PATH", "")).expanduser() if __import__("os").environ.get("E3LAB_FEATURE_SETS_PATH") else None,
            "feature_sets_tfm_1h.json",
            "data/feature_sets_tfm_1h.json",
            "datasets/feature_sets_tfm_1h.json",
            BASE_DIR.parent / "feature_sets_tfm_1h.json",
            BASE_DIR.parent / "data" / "feature_sets_tfm_1h.json",
            BASE_DIR.parent / "datasets" / "feature_sets_tfm_1h.json",
        ],
        filename="feature_sets_tfm_1h.json",
        search_roots=[BASE_DIR, BASE_DIR.parent],
    )

    OUTPUT_DIR = BASE_DIR / "Graficos" / "xgb_direct24_covariables_futuras"
    MODEL_DIR = BASE_DIR / "modelos_guardados_xgb_direct24_covariables_futuras"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    TARGET_HOURLY = "target_energia_1h"
    HORIZON = 24
    TRAIN_STRIDE = 6
    EVAL_STRIDE = 24
    REPRESENTATIVE_HORIZONS = [1, 2, 4, 8, 12, 18, 24]

    if DATA_PATH is None:
        raise FileNotFoundError(
            "No se encontró E3Lab_modelo_1h.csv. Puedes dejarlo en la raíz del proyecto, en una carpeta data/ o definir la variable de entorno E3LAB_DATASET_PATH."
        )
    if FEATURE_SETS_PATH is None:
        raise FileNotFoundError(
            "No se encontró feature_sets_tfm_1h.json. Puedes dejarlo en la raíz del proyecto, en una carpeta data/ o definir la variable de entorno E3LAB_FEATURE_SETS_PATH."
        )

    print(f"Dataset: {DATA_PATH.resolve()}")
    print(f"Feature sets: {FEATURE_SETS_PATH.resolve()}")
    print(f"Horizonte: {HORIZON}")
    print(f"TRAIN_STRIDE={TRAIN_STRIDE} | EVAL_STRIDE={EVAL_STRIDE}")
    if torch is None:
        print(f"PyTorch no disponible: {TORCH_IMPORT_ERROR}")
    else:
        print(f"PyTorch: {torch.__version__} | device={DEVICE}")


## 1. Carga de datos y entendimiento inicial

En esta primera parte cargamos el dataset horario generado en ETL y validamos que el notebook está leyendo la versión correcta del problema:

- dataset base: `E3Lab_modelo_1h.csv`
- target: `target_energia_1h`
- frecuencia de modelado: 1 hora

Aquí todavía no entrenamos nada. La idea es asegurarnos de que:

- la granularidad es la correcta,
- las variables clave existen,
- y el punto de partida del modelado es consistente con el objetivo de predecir 24 horas completas.
    


In [ ]:
import json

df_hourly = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True).sort_index()
df_hourly = df_hourly[~df_hourly.index.duplicated(keep="first")]

with open(FEATURE_SETS_PATH, "r", encoding="utf-8") as f:
    feature_sets_1h = json.load(f)

print(f"Shape horario: {df_hourly.shape}")
print(f"Periodo horario: {df_hourly.index.min()} -> {df_hourly.index.max()}")
print(df_hourly[TARGET_HOURLY].describe().round(4))
print("Feature sets cargados desde ETL:")
print(list(feature_sets_1h.keys()))

## 2. EDA: qué estamos modelando realmente

Antes de entrar al feature engineering, conviene mirar el problema desde tres perspectivas simples:

- **escala y calidad del dataset**,
- **perfil temporal del consumo**,
- **relación entre el target y algunas variables de contexto**.

No buscamos un EDA exhaustivo, sino uno útil para entender por qué este problema es difícil y dónde pueden estar las señales más relevantes.
    


In [ ]:
    print("=== RESUMEN INICIAL DEL DATASET HORARIO ===")
    resumen_eda = pd.DataFrame(
        {
            "n_filas": [len(df_hourly)],
            "n_columnas": [df_hourly.shape[1]],
            "fecha_inicio": [df_hourly.index.min()],
            "fecha_fin": [df_hourly.index.max()],
            "missing_total": [int(df_hourly.isna().sum().sum())],
            "target_mean": [float(df_hourly[TARGET_HOURLY].mean())],
            "target_std": [float(df_hourly[TARGET_HOURLY].std())],
        }
    )
    display(resumen_eda)

    top_missing = (
        df_hourly.isna().mean()
        .sort_values(ascending=False)
        .head(12)
        .rename("missing_pct")
        .mul(100)
        .reset_index()
        .rename(columns={"index": "variable"})
    )
    display(top_missing)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    axes[0, 0].plot(df_hourly.index, df_hourly[TARGET_HOURLY], color="#2a6f97", linewidth=0.7)
    axes[0, 0].set_title("Serie temporal del consumo HVAC horario")
    axes[0, 0].set_ylabel("kWh")

    perfil_hora = df_hourly.groupby("hour")[TARGET_HOURLY].mean()
    axes[0, 1].plot(perfil_hora.index, perfil_hora.values, marker="o", color="#d62828", linewidth=2)
    axes[0, 1].axvspan(8, 10, color="#fcbf49", alpha=0.18)
    axes[0, 1].set_title("Perfil medio por hora del día")
    axes[0, 1].set_xlabel("Hora")
    axes[0, 1].set_ylabel("kWh medio")

    if "day_of_week" in df_hourly.columns:
        perfil_dow = df_hourly.groupby("day_of_week")[TARGET_HOURLY].mean()
        axes[1, 0].bar(perfil_dow.index.astype(str), perfil_dow.values, color="#6a994e", alpha=0.85)
        axes[1, 0].set_title("Perfil medio por día de la semana")
        axes[1, 0].set_ylabel("kWh medio")

    if "meteo_temp_ext" in df_hourly.columns:
        sample_scatter = df_hourly[[TARGET_HOURLY, "meteo_temp_ext"]].dropna().sample(
            min(3000, len(df_hourly.dropna(subset=[TARGET_HOURLY, "meteo_temp_ext"]))),
            random_state=SEED,
        )
        axes[1, 1].scatter(
            sample_scatter["meteo_temp_ext"],
            sample_scatter[TARGET_HOURLY],
            s=12,
            alpha=0.25,
            color="#7b2cbf",
        )
        axes[1, 1].set_title("Consumo vs temperatura exterior")
        axes[1, 1].set_xlabel("Temperatura exterior")
        axes[1, 1].set_ylabel("kWh")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "eda_direct24_resumen.png", bbox_inches="tight")
    plt.show()
    


**Lectura rápida del EDA**

- El target presenta una estructura horaria clara, pero no completamente estable.
- La franja de arranque del edificio aparece como una zona especialmente sensible.
- La temperatura exterior tiene relación con el consumo, pero claramente no explica por sí sola toda la variabilidad.
- Desde el principio se intuye que este no es un problema solo meteorológico; también hay una componente operativa importante.
    


## 3. Feature engineering: regímenes, variables de arranque y memoria temporal

Antes de construir el problema `Direct-24`, organizamos las variables que más sentido tienen para un edificio con comportamiento operativo heterogéneo.

En esta sección se incorporan tres ideas importantes:

- **Regímenes operativos**: `heating`, `transition` y `low_demand`
- **Variables de arranque**: para intentar capturar mejor la franja crítica de 8:00 a 10:00
- **Memoria estructural**: misma hora del día anterior, de hace 2 días, de la semana anterior y medias por misma hora

Más adelante veremos si estas señales realmente ayudan a mejorar el desempeño.
    


In [ ]:
    def assign_regime(month):
        if month in [11, 12, 1, 2, 3]:
            return "heating"
        if month in [4, 5, 10]:
            return "transition"
        return "low_demand"

    if "month" not in df_hourly.columns:
        raise ValueError("El ETL horario debe contener la columna 'month'.")
    df_hourly["regime"] = df_hourly["month"].map(assign_regime)

    required_cols = ["hour", "day_of_week", "meteo_temp_ext", "calendario_lectivo", TARGET_HOURLY]
    missing = [c for c in required_cols if c not in df_hourly.columns]
    if missing:
        raise ValueError(f"Faltan columnas clave del ETL horario: {missing}")

    # Features específicas para capturar el encendido del edificio.
    df_hourly["is_opening_window"] = df_hourly["hour"].between(8, 10).astype(int)
    df_hourly["is_pre_opening"] = df_hourly["hour"].between(6, 7).astype(int)
    df_hourly["is_monday"] = (df_hourly["day_of_week"] == 0).astype(int)
    df_hourly["opening_x_lectivo"] = df_hourly["is_opening_window"] * df_hourly["calendario_lectivo"]
    df_hourly["opening_temp_interaction"] = df_hourly["is_opening_window"] * df_hourly["meteo_temp_ext"]

    # Paso 3: memoria estructural fuerte de la misma hora.
    df_hourly["target_same_hour_yesterday"] = df_hourly[TARGET_HOURLY].shift(24)
    df_hourly["target_same_hour_2days"] = df_hourly[TARGET_HOURLY].shift(48)
    df_hourly["target_same_hour_last_week"] = df_hourly[TARGET_HOURLY].shift(168)

    same_hour_lag_1d = df_hourly[TARGET_HOURLY].shift(24)
    same_hour_lag_2d = df_hourly[TARGET_HOURLY].shift(48)
    same_hour_lag_3d = df_hourly[TARGET_HOURLY].shift(72)
    same_hour_lag_4d = df_hourly[TARGET_HOURLY].shift(96)
    same_hour_lag_5d = df_hourly[TARGET_HOURLY].shift(120)
    same_hour_lag_6d = df_hourly[TARGET_HOURLY].shift(144)
    same_hour_lag_7d = df_hourly[TARGET_HOURLY].shift(168)

    df_hourly["target_same_hour_mean_3d"] = pd.concat(
        [same_hour_lag_1d, same_hour_lag_2d, same_hour_lag_3d], axis=1
    ).mean(axis=1)
    df_hourly["target_same_hour_mean_7d"] = pd.concat(
        [
            same_hour_lag_1d,
            same_hour_lag_2d,
            same_hour_lag_3d,
            same_hour_lag_4d,
            same_hour_lag_5d,
            same_hour_lag_6d,
            same_hour_lag_7d,
        ],
        axis=1,
    ).mean(axis=1)

    # Paso 4: referencias meteorologicas para construir diferencias futuras.
    df_hourly["meteo_temp_same_hour_yesterday"] = df_hourly["meteo_temp_ext"].shift(24)
    df_hourly["meteo_temp_same_hour_2days"] = df_hourly["meteo_temp_ext"].shift(48)
    df_hourly["meteo_temp_same_hour_last_week"] = df_hourly["meteo_temp_ext"].shift(168)

    print(df_hourly["regime"].value_counts().to_string())
    print("Variables cronológicas verificadas: hour, day_of_week")
    print("Features de arranque, memoria y referencias meteo añadidas:")
    print([
        "is_opening_window",
        "is_pre_opening",
        "is_monday",
        "opening_x_lectivo",
        "opening_temp_interaction",
        "target_same_hour_yesterday",
        "target_same_hour_2days",
        "target_same_hour_last_week",
        "target_same_hour_mean_3d",
        "target_same_hour_mean_7d",
        "meteo_temp_same_hour_yesterday",
        "meteo_temp_same_hour_2days",
        "meteo_temp_same_hour_last_week",
    ])
    


## 4. Selección de features y lectura del trabajo previo de ETL

Aquí conectamos el notebook con los `feature_sets` definidos en ETL y extendemos ese conjunto con variables que fueron apareciendo durante el diagnóstico:

- features históricas del origen `t`
- covariables futuras alineadas por horizonte `t+h`
- variables de memoria explícita
- señales de arranque del edificio

La lógica es conservar el trabajo previo útil, pero dejar muy claro qué entra al modelo y por qué.
    


In [ ]:
    df_model = df_hourly.copy()

    HISTORICAL_FEATURES = [c for c in feature_sets_1h["xgb_direct24_hist_1h"] if c in df_model.columns]
    FUTURE_KNOWN_FEATURES = [c for c in feature_sets_1h["xgb_direct24_future_1h"] if c in df_model.columns]

    EXTRA_HIST_FEATURES = [
        "is_opening_window",
        "is_pre_opening",
        "is_monday",
        "opening_x_lectivo",
        "opening_temp_interaction",
    ]

    # Paso 3 y 4: memoria explicita y referencias futuras conocidas.
    EXTRA_FUTURE_FEATURES = [
        "is_opening_window",
        "is_pre_opening",
        "is_monday",
        "opening_x_lectivo",
        "opening_temp_interaction",
        "target_same_hour_yesterday",
        "target_same_hour_2days",
        "target_same_hour_last_week",
        "target_same_hour_mean_3d",
        "target_same_hour_mean_7d",
        "meteo_temp_same_hour_yesterday",
        "meteo_temp_same_hour_2days",
        "meteo_temp_same_hour_last_week",
    ]

    HISTORICAL_FEATURES = list(dict.fromkeys(HISTORICAL_FEATURES + [c for c in EXTRA_HIST_FEATURES if c in df_model.columns]))
    FUTURE_KNOWN_FEATURES = list(dict.fromkeys(FUTURE_KNOWN_FEATURES + [c for c in EXTRA_FUTURE_FEATURES if c in df_model.columns]))

    print(f"Features históricas: {len(HISTORICAL_FEATURES)}")
    print(f"Features futuras conocidas: {len(FUTURE_KNOWN_FEATURES)}")
    print("Extras históricas activas:", [c for c in EXTRA_HIST_FEATURES if c in HISTORICAL_FEATURES])
    print("Extras futuras activas:", [c for c in EXTRA_FUTURE_FEATURES if c in FUTURE_KNOWN_FEATURES])
    


## 5. Limpieza, split temporal e imputación causal

Como se trata de una serie temporal, el particionado y la imputación deben respetar estrictamente la causalidad.

Decisiones clave:

- `train / val / test` cronológico
- imputación usando únicamente pasado disponible
- medianas aprendidas solo en `train`

Esta parte es especialmente importante porque evita fugas de información y hace que la evaluación sea defendible.
    


In [ ]:
df_clean = df_model.dropna(subset=[TARGET_HOURLY]).copy()

n = len(df_clean)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

df_train_raw = df_clean.iloc[:n_train].copy()
df_val_raw = df_clean.iloc[n_train:n_train + n_val].copy()
df_test_raw = df_clean.iloc[n_train + n_val:].copy()

IMPUTE_FEATURES = sorted(set(HISTORICAL_FEATURES) | set(FUTURE_KNOWN_FEATURES))
train_feature_medians = df_train_raw[IMPUTE_FEATURES].median(numeric_only=True)

def imputar_segmento(df_segmento, contexto_previo=None):
    if contexto_previo is not None and len(contexto_previo) > 0:
        combinado = pd.concat([contexto_previo.tail(1), df_segmento], axis=0)
        features_imp = combinado[IMPUTE_FEATURES].ffill().iloc[1:]
    else:
        features_imp = df_segmento[IMPUTE_FEATURES].ffill()

    out = df_segmento.copy()
    out[IMPUTE_FEATURES] = features_imp.fillna(train_feature_medians)
    return out

df_train = imputar_segmento(df_train_raw)
df_val = imputar_segmento(df_val_raw, contexto_previo=df_train)
df_test = imputar_segmento(df_test_raw, contexto_previo=df_val)

print(f"Train: {df_train.index.min()} -> {df_train.index.max()} | filas={len(df_train):,}")
print(f"Val:   {df_val.index.min()} -> {df_val.index.max()} | filas={len(df_val):,}")
print(f"Test:  {df_test.index.min()} -> {df_test.index.max()} | filas={len(df_test):,}")

## 6. Construcción del problema Direct-24

En lugar de predecir un único punto futuro, aquí construimos el bloque completo de 24 horas.

Para cada origen temporal `t`:

- `X(t)` contiene historia disponible hasta el origen
- `Y(t)` contiene el vector `t+1 ... t+24`
- además se incorporan covariables futuras conocidas alineadas al horizonte

Esta es la pieza metodológica central del notebook.
    


In [ ]:
    def contiguous_origin_mask(index, horizon, freq="1h"):
        step = pd.Timedelta(freq)
        diff_ok = pd.Series(pd.Index(index)).diff().eq(step).fillna(False).to_numpy().astype(int)
        rolling = pd.Series(diff_ok).rolling(horizon, min_periods=horizon).sum().to_numpy()
        return rolling[horizon:] == horizon

    def build_direct_datasets(df_train, df_val, df_test, target, horizon=24, train_stride=6, eval_stride=24):
        df_all = pd.concat(
            [df_train.assign(_split="train"), df_val.assign(_split="val"), df_test.assign(_split="test")]
        ).sort_index()

        n_all = len(df_all)
        n_samples = n_all - horizon
        origin_index = df_all.index[:n_samples]
        split_values = df_all["_split"].to_numpy()
        origin_split = split_values[:n_samples]
        y_values = df_all[target].to_numpy(dtype=float)
        Y_all = np.column_stack([y_values[h:n_samples + h] for h in range(1, horizon + 1)])
        regime_origin = df_all["regime"].iloc[:n_samples].to_numpy()

        same_split_mask = np.ones(n_samples, dtype=bool)
        for h in range(1, horizon + 1):
            same_split_mask &= origin_split == split_values[h:n_samples + h]

        valid_mask = (
            same_split_mask
            & contiguous_origin_mask(df_all.index, horizon, freq="1h")
            & np.isfinite(Y_all).all(axis=1)
        )

        def positions(split_name, stride):
            pos = np.where(valid_mask & (origin_split == split_name))[0]
            return pos[::stride]

        pos_train = positions("train", train_stride)
        pos_val = positions("val", eval_stride)
        pos_test = positions("test", eval_stride)

        X_hist_base = df_all[HISTORICAL_FEATURES].iloc[:n_samples].reset_index(drop=True)
        direct = {}

        for h in range(1, horizon + 1):
            future_h = df_all[FUTURE_KNOWN_FEATURES].iloc[h:n_samples + h].reset_index(drop=True).copy()
            future_h.columns = [f"{c}__t_plus_{h}" for c in FUTURE_KNOWN_FEATURES]

            # Paso 4: diferencias futuras que aportan cambio relativo y no solo nivel absoluto.
            if "meteo_temp_ext__t_plus_{}".format(h) in future_h.columns and "meteo_temp_ext" in X_hist_base.columns:
                future_h[f"temp_future_minus_now__t_plus_{h}"] = (
                    future_h[f"meteo_temp_ext__t_plus_{h}"] - X_hist_base["meteo_temp_ext"]
                )

            if "meteo_hr_ext__t_plus_{}".format(h) in future_h.columns and "meteo_hr_ext" in X_hist_base.columns:
                future_h[f"humidity_future_minus_now__t_plus_{h}"] = (
                    future_h[f"meteo_hr_ext__t_plus_{h}"] - X_hist_base["meteo_hr_ext"]
                )

            if "meteo_solar_rad__t_plus_{}".format(h) in future_h.columns and "meteo_solar_rad" in X_hist_base.columns:
                future_h[f"solar_future_minus_now__t_plus_{h}"] = (
                    future_h[f"meteo_solar_rad__t_plus_{h}"] - X_hist_base["meteo_solar_rad"]
                )

            if (
                f"meteo_temp_ext__t_plus_{h}" in future_h.columns
                and f"meteo_temp_same_hour_yesterday__t_plus_{h}" in future_h.columns
            ):
                future_h[f"temp_future_minus_yesterday_same_hour__t_plus_{h}"] = (
                    future_h[f"meteo_temp_ext__t_plus_{h}"] - future_h[f"meteo_temp_same_hour_yesterday__t_plus_{h}"]
                )

            if (
                f"meteo_temp_ext__t_plus_{h}" in future_h.columns
                and "temp_int_media" in X_hist_base.columns
            ):
                future_h[f"temp_gap_future__t_plus_{h}"] = (
                    X_hist_base["temp_int_media"] - future_h[f"meteo_temp_ext__t_plus_{h}"]
                )

            X_h = pd.concat([X_hist_base, future_h], axis=1)
            X_h.index = origin_index

            finite_x = np.isfinite(X_h.to_numpy(dtype=float)).all(axis=1)
            direct[h] = {
                "X_train": X_h.iloc[pos_train[finite_x[pos_train]]].copy(),
                "X_val": X_h.iloc[pos_val[finite_x[pos_val]]].copy(),
                "X_test": X_h.iloc[pos_test[finite_x[pos_test]]].copy(),
            }

        return {
            "df_all": df_all.drop(columns="_split"),
            "origin_index": origin_index,
            "regime_origin": regime_origin,
            "Y_train": Y_all[pos_train].copy(),
            "Y_val": Y_all[pos_val].copy(),
            "Y_test": Y_all[pos_test].copy(),
            "idx_train": origin_index[pos_train],
            "idx_val": origin_index[pos_val],
            "idx_test": origin_index[pos_test],
            "regime_train": regime_origin[pos_train],
            "regime_val": regime_origin[pos_val],
            "regime_test": regime_origin[pos_test],
            "direct": direct,
        }

    data = build_direct_datasets(
        df_train=df_train,
        df_val=df_val,
        df_test=df_test,
        target=TARGET_HOURLY,
        horizon=HORIZON,
        train_stride=TRAIN_STRIDE,
        eval_stride=EVAL_STRIDE,
    )

    Y_train = data["Y_train"]
    Y_val = data["Y_val"]
    Y_test = data["Y_test"]

    print(f"Muestras diarias train={len(Y_train):,} | val={len(Y_val):,} | test={len(Y_test):,}")
    print("Ejemplo de variables futuras alineadas en h=4:")
    print([c for c in data["direct"][4]["X_train"].columns if "__t_plus_4" in c][:20])
    


## 7. Modelado base: XGBoost Direct-24 global

Empezamos por el modelo global porque sirve como referencia estructural:

- un único esquema de entrenamiento para los 24 horizontes
- misma lógica para todo el año
- buen punto de comparación frente a variantes más especializadas

Aquí lo que intentamos validar es si el patrón general del edificio puede modelarse sin separar contextos operativos.
    


In [ ]:
def clip_predictions(y_pred):
    return np.maximum(np.asarray(y_pred, dtype=float), 0.0)

def evaluate_multistep(y_true, y_pred, model_name):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = clip_predictions(y_pred)

    mae_h = np.mean(np.abs(y_true - y_pred), axis=0)
    rmse_h = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))
    r2_h = np.array([r2_score(y_true[:, h], y_pred[:, h]) for h in range(y_true.shape[1])])

    return {
        "modelo": model_name,
        "MAE": float(mae_h.mean()),
        "RMSE": float(rmse_h.mean()),
        "R2": float(np.nanmean(r2_h)),
        "WAPE": float((np.abs(y_true - y_pred).sum() / (np.abs(y_true).sum() + 1e-8)) * 100),
        "sMAPE": float(np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100),
        "RMSE_h1": float(rmse_h[0]),
        "RMSE_h24": float(rmse_h[-1]),
        "rmse_horizonte": rmse_h,
    }

xgb_grid = [
    {"max_depth": 3, "learning_rate": 0.03, "min_child_weight": 8, "subsample": 0.80, "colsample_bytree": 0.75, "reg_alpha": 0.2, "reg_lambda": 4.0},
    {"max_depth": 4, "learning_rate": 0.025, "min_child_weight": 10, "subsample": 0.75, "colsample_bytree": 0.70, "reg_alpha": 0.3, "reg_lambda": 5.0},
    {"max_depth": 4, "learning_rate": 0.02, "min_child_weight": 12, "subsample": 0.70, "colsample_bytree": 0.65, "reg_alpha": 0.4, "reg_lambda": 6.0},
]

best_params_global = None
best_score_global = np.inf
global_rows = []

for params in xgb_grid:
    rmse_list = []
    for h in REPRESENTATIVE_HORIZONS:
        model = xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=300,
            tree_method="hist",
            random_state=SEED + h,
            n_jobs=-1,
            **params,
        )
        model.fit(data["direct"][h]["X_train"].values, Y_train[:, h - 1])
        pred_val_h = clip_predictions(model.predict(data["direct"][h]["X_val"].values))
        rmse_h = float(np.sqrt(np.mean((Y_val[:, h - 1] - pred_val_h) ** 2)))
        rmse_list.append(rmse_h)

    mean_rmse = float(np.mean(rmse_list))
    global_rows.append({**params, "RMSE_val_repr": mean_rmse})
    print(f"Global params={params} | mean_RMSE_val_repr={mean_rmse:.4f}")
    if mean_rmse < best_score_global:
        best_score_global = mean_rmse
        best_params_global = params

print("Mejores parametros globales:", best_params_global)

## 8. XGBoost Direct-24 por régimen

Después del modelo global, probamos una variante por régimen operativo.

La hipótesis detrás de esta comparación es sencilla:

- el edificio no se comporta igual en meses fríos, meses de transición y periodos de baja demanda.
- por tanto, separar el aprendizaje puede reducir el promedio forzado entre dinámicas muy distintas

Más adelante veremos si esa segmentación realmente mejora métricas o solo añade complejidad.
    


In [ ]:
regime_order = ["heating", "transition", "low_demand"]

Y_dev = np.vstack([Y_train, Y_val])
regime_dev = np.concatenate([data["regime_train"], data["regime_val"]])
direct_dev = {
    h: pd.concat([data["direct"][h]["X_train"], data["direct"][h]["X_val"]], axis=0).reset_index(drop=True)
    for h in range(1, HORIZON + 1)
}

def split_dev_positions_by_regime(regime_name, val_fraction=0.20):
    pos = np.where(pd.Series(regime_dev).eq(regime_name).to_numpy())[0]
    if len(pos) < 40:
        return None, None
    cut = max(int(len(pos) * (1 - val_fraction)), 30)
    cut = min(cut, len(pos) - 5)
    return pos[:cut], pos[cut:]

def tune_regime_xgb(regime_name):
    dev_train_pos, dev_val_pos = split_dev_positions_by_regime(regime_name)
    best_params = None
    best_score = np.inf
    rows = []

    for params in xgb_grid:
        rmse_list = []
        valid_h_count = 0
        for h in REPRESENTATIVE_HORIZONS:
            if dev_train_pos is None or dev_val_pos is None:
                continue

            X_train_h = direct_dev[h].iloc[dev_train_pos].values
            y_train_h = Y_dev[dev_train_pos, h - 1]
            X_val_h = direct_dev[h].iloc[dev_val_pos].values
            y_val_h = Y_dev[dev_val_pos, h - 1]

            if len(X_train_h) < 30 or len(X_val_h) < 5:
                continue

            model = xgb.XGBRegressor(
                objective="reg:squarederror",
                n_estimators=300,
                tree_method="hist",
                random_state=SEED + h,
                n_jobs=-1,
                **params,
            )
            model.fit(X_train_h, y_train_h)
            pred_val_h = clip_predictions(model.predict(X_val_h))
            rmse_h = float(np.sqrt(np.mean((y_val_h - pred_val_h) ** 2)))
            rmse_list.append(rmse_h)
            valid_h_count += 1

        score = float(np.mean(rmse_list)) if rmse_list else np.inf
        rows.append({**params, "regime": regime_name, "RMSE_val_repr": score, "valid_horizons": valid_h_count})
        print(f"Regime={regime_name} params={params} | mean_RMSE_val_repr={score:.4f} | valid_h={valid_h_count}")
        if score < best_score:
            best_score = score
            best_params = params

    return best_params, rows

regime_best_params = {}
regime_rows = []
for regime_name in regime_order:
    best_params, rows = tune_regime_xgb(regime_name)
    regime_best_params[regime_name] = best_params
    regime_rows.extend(rows)
    print(f"Mejores parametros para {regime_name}: {best_params}")

## 9. Primer bloque de resultados: global vs por régimen

En este punto comparamos las dos variantes principales de XGBoost:

- modelo global
- modelo por régimen

Esta comparación no cierra todavía el análisis, pero sí establece una línea base sólida para todo lo que viene después.
    


In [ ]:
# Modelo global
pred_global_cols = []
for h in range(1, HORIZON + 1):
    X_fit_h = pd.concat([data["direct"][h]["X_train"], data["direct"][h]["X_val"]], axis=0)
    y_fit_h = np.concatenate([Y_train[:, h - 1], Y_val[:, h - 1]])
    model_h = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        tree_method="hist",
        random_state=SEED + h,
        n_jobs=-1,
        **best_params_global,
    )
    model_h.fit(X_fit_h.values, y_fit_h)
    pred_h = clip_predictions(model_h.predict(data["direct"][h]["X_test"].values))
    pred_global_cols.append(pred_h)

pred_global = np.column_stack(pred_global_cols)
res_global = evaluate_multistep(Y_test, pred_global, "XGBoost Direct-24 Global")

# Modelo por régimen
pred_regime_cols = []
for h in range(1, HORIZON + 1):
    pred_h = np.zeros(len(Y_test), dtype=float)

    for regime_name in regime_order:
        test_mask = pd.Series(data["regime_test"]).eq(regime_name).to_numpy()
        params = regime_best_params[regime_name]
        if params is None or test_mask.sum() == 0:
            continue

        dev_pos = np.where(pd.Series(regime_dev).eq(regime_name).to_numpy())[0]
        X_fit_h = direct_dev[h].iloc[dev_pos]
        y_fit_h = Y_dev[dev_pos, h - 1]
        if len(X_fit_h) < 30:
            continue

        model_h = xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=300,
            tree_method="hist",
            random_state=SEED + 100 + h,
            n_jobs=-1,
            **params,
        )
        model_h.fit(X_fit_h.values, y_fit_h)
        pred_h_regime = clip_predictions(model_h.predict(data["direct"][h]["X_test"].iloc[test_mask].values))
        pred_h[test_mask] = pred_h_regime

    pred_regime_cols.append(pred_h)

pred_regime = np.column_stack(pred_regime_cols)
res_regime = evaluate_multistep(Y_test, pred_regime, "XGBoost Direct-24 por Regimen")

print(pd.DataFrame([res_global, res_regime]).drop(columns=["rmse_horizonte"]).round(4).to_string(index=False))

## 10. Exportación intermedia de resultados base

Guardamos aquí los artefactos del bloque base de XGBoost para no perder trazabilidad:

- búsqueda global
- búsqueda por régimen
- comparativa base
    


In [ ]:
pd.DataFrame(global_rows).to_csv(MODEL_DIR / "xgb_global_search_results.csv", index=False)
pd.DataFrame(regime_rows).to_csv(MODEL_DIR / "xgb_regime_search_results.csv", index=False)
pd.DataFrame(
    [
        {k: v for k, v in res_global.items() if k != "rmse_horizonte"},
        {k: v for k, v in res_regime.items() if k != "rmse_horizonte"},
    ]
).to_csv(MODEL_DIR / "comparativa_xgb_direct24_covariables.csv", index=False)

print("Archivos exportados:")
print(MODEL_DIR / "xgb_global_search_results.csv")
print(MODEL_DIR / "xgb_regime_search_results.csv")
print(MODEL_DIR / "comparativa_xgb_direct24_covariables.csv")

## 11. Evaluación detallada y auditoría del pipeline

Antes de seguir añadiendo mejoras, esta sección audita el notebook desde varios ángulos:

- qué variables se usan realmente,
- qué cobertura tienen `train / val / test`,
- cómo se degrada el modelo por horizonte,
- qué señales parecen dominar el aprendizaje.

La idea es evitar una optimización ciega y entender mejor por qué el modelo está funcionando como está funcionando.
    


In [ ]:
        used_features = sorted(set(HISTORICAL_FEATURES) | set(FUTURE_KNOWN_FEATURES))
        csv_columns = [c for c in df_hourly.columns]
        unused_columns = [c for c in csv_columns if c not in used_features and c not in [TARGET_HOURLY, "regime"]]

        print("=== AUDITORIA DE VARIABLES ===")
        print(f"Columnas totales del CSV horario: {len(csv_columns)}")
        print(f"Features historicas usadas: {len(HISTORICAL_FEATURES)}")
        print(f"Features futuras usadas: {len(FUTURE_KNOWN_FEATURES)}")
        print(f"Union de features efectivamente usadas: {len(used_features)}")
        print()

        print("Primeras 20 features historicas:")
        print(HISTORICAL_FEATURES[:20])
        print()
        print("Primeras 20 features futuras:")
        print(FUTURE_KNOWN_FEATURES[:20])
        print()
        print("Columnas presentes en el CSV pero no usadas por el modelo:")
        print(unused_columns[:30])

        audit_rows = []
        for col in used_features:
            audit_rows.append(
                {
                    "feature": col,
                    "usada_como_historica": int(col in HISTORICAL_FEATURES),
                    "usada_como_futura": int(col in FUTURE_KNOWN_FEATURES),
                    "missing_pct_csv": float(df_hourly[col].isna().mean() * 100) if col in df_hourly.columns else np.nan,
                    "dtype_csv": str(df_hourly[col].dtype) if col in df_hourly.columns else "missing",
                }
            )

        df_feature_audit = pd.DataFrame(audit_rows).sort_values(
            ["missing_pct_csv", "feature"], ascending=[False, True]
        )
        display(df_feature_audit.head(25))

        feature_overlap = pd.DataFrame(
            {
                "grupo": ["historicas", "futuras", "interseccion", "solo_historicas", "solo_futuras"],
                "n_features": [
                    len(HISTORICAL_FEATURES),
                    len(FUTURE_KNOWN_FEATURES),
                    len(set(HISTORICAL_FEATURES) & set(FUTURE_KNOWN_FEATURES)),
                    len(set(HISTORICAL_FEATURES) - set(FUTURE_KNOWN_FEATURES)),
                    len(set(FUTURE_KNOWN_FEATURES) - set(HISTORICAL_FEATURES)),
                ],
            }
        )
        display(feature_overlap)
        


In [ ]:
        print("=== COBERTURA TEMPORAL Y POR REGIMEN ===")
        split_summary = pd.DataFrame(
            [
                {
                    "split": "train",
                    "inicio": df_train.index.min(),
                    "fin": df_train.index.max(),
                    "filas_horarias": len(df_train),
                    "muestras_origen_direct24": len(Y_train),
                },
                {
                    "split": "val",
                    "inicio": df_val.index.min(),
                    "fin": df_val.index.max(),
                    "filas_horarias": len(df_val),
                    "muestras_origen_direct24": len(Y_val),
                },
                {
                    "split": "test",
                    "inicio": df_test.index.min(),
                    "fin": df_test.index.max(),
                    "filas_horarias": len(df_test),
                    "muestras_origen_direct24": len(Y_test),
                },
            ]
        )
        display(split_summary)

        regime_split = pd.concat(
            [
                df_train.assign(split="train"),
                df_val.assign(split="val"),
                df_test.assign(split="test"),
            ]
        )
        regime_counts = (
            regime_split.groupby(["split", "regime"])
            .size()
            .rename("filas")
            .reset_index()
            .pivot(index="regime", columns="split", values="filas")
            .fillna(0)
            .astype(int)
        )
        display(regime_counts)

        origin_regime_counts = pd.DataFrame(
            {
                "train": pd.Series(data["regime_train"]).value_counts(),
                "val": pd.Series(data["regime_val"]).value_counts(),
                "test": pd.Series(data["regime_test"]).value_counts(),
            }
        ).fillna(0).astype(int)
        display(origin_regime_counts)

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        regime_counts.plot(kind="bar", ax=axes[0], color=["#4c78a8", "#f58518", "#54a24b"])
        axes[0].set_title("Cobertura horaria por regimen y split")
        axes[0].set_ylabel("Numero de filas")
        axes[0].tick_params(axis="x", rotation=0)

        origin_regime_counts.plot(kind="bar", ax=axes[1], color=["#4c78a8", "#f58518", "#54a24b"])
        axes[1].set_title("Orígenes Direct-24 validos por regimen")
        axes[1].set_ylabel("Numero de muestras")
        axes[1].tick_params(axis="x", rotation=0)

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "auditoria_cobertura_regimen_split.png", bbox_inches="tight")
        plt.show()
        


In [ ]:
        print("=== DIAGNOSTICO DE RENDIMIENTO POR HORIZONTE ===")
        rmse_global = np.asarray(res_global["rmse_horizonte"], dtype=float)
        rmse_regime = np.asarray(res_regime["rmse_horizonte"], dtype=float)
        r2_global_h = np.array([r2_score(Y_test[:, h], pred_global[:, h]) for h in range(HORIZON)])
        r2_regime_h = np.array([r2_score(Y_test[:, h], pred_regime[:, h]) for h in range(HORIZON)])

        df_h = pd.DataFrame(
            {
                "horizonte_horas": np.arange(1, HORIZON + 1),
                "RMSE_global": rmse_global,
                "RMSE_regime": rmse_regime,
                "R2_global": r2_global_h,
                "R2_regime": r2_regime_h,
                "mean_target_real": Y_test.mean(axis=0),
            }
        )
        display(df_h.head(24))

        fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

        axes[0].plot(df_h["horizonte_horas"], df_h["RMSE_global"], marker="o", linewidth=2.0, color="#e45756", label="Global")
        axes[0].plot(df_h["horizonte_horas"], df_h["RMSE_regime"], marker="o", linewidth=2.0, color="#4c78a8", label="Por regimen")
        axes[0].fill_between(df_h["horizonte_horas"], df_h["RMSE_global"], color="#e45756", alpha=0.12)
        axes[0].fill_between(df_h["horizonte_horas"], df_h["RMSE_regime"], color="#4c78a8", alpha=0.10)
        axes[0].set_title("Degradacion del error por horizonte")
        axes[0].set_ylabel("RMSE")
        axes[0].legend()

        axes[1].plot(df_h["horizonte_horas"], df_h["R2_global"], marker="o", linewidth=2.0, color="#e45756", label="Global")
        axes[1].plot(df_h["horizonte_horas"], df_h["R2_regime"], marker="o", linewidth=2.0, color="#4c78a8", label="Por regimen")
        axes[1].axhline(0, linestyle="--", linewidth=1.2, color="black", alpha=0.7)
        axes[1].set_title("Capacidad explicativa por horizonte")
        axes[1].set_xlabel("Horizonte (horas)")
        axes[1].set_ylabel("R2")
        axes[1].legend()

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "auditoria_rmse_r2_por_horizonte.png", bbox_inches="tight")
        plt.show()

        df_compare = pd.DataFrame(
            [
                {k: v for k, v in res_global.items() if k != "rmse_horizonte"},
                {k: v for k, v in res_regime.items() if k != "rmse_horizonte"},
            ]
        ).set_index("modelo")

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        metric_specs = [("RMSE", "#e45756"), ("R2", "#54a24b"), ("sMAPE", "#f58518")]
        for ax, (metric, color) in zip(axes, metric_specs):
            vals = df_compare[metric].sort_values(ascending=(metric != "R2"))
            bars = ax.bar(vals.index, vals.values, color=color, alpha=0.85, edgecolor="white")
            ax.set_title(metric)
            ax.tick_params(axis="x", rotation=15)
            offset = max(np.nanmax(np.abs(vals.values)) * 0.02, 1e-6)
            for bar, val in zip(bars, vals.values):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + offset, f"{val:.3f}", ha="center", fontsize=8)

        plt.suptitle("Comparativa final del notebook")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "auditoria_comparativa_global_vs_regimen.png", bbox_inches="tight")
        plt.show()
        


In [ ]:
        print("=== IMPORTANCIA DE VARIABLES EN HORIZONTES CLAVE ===")
        audit_horizons = [1, 4, 12, 24]
        importance_frames = []

        for h in audit_horizons:
            X_fit_h = pd.concat([data["direct"][h]["X_train"], data["direct"][h]["X_val"]], axis=0)
            y_fit_h = np.concatenate([Y_train[:, h - 1], Y_val[:, h - 1]])

            model_h = xgb.XGBRegressor(
                objective="reg:squarederror",
                n_estimators=300,
                tree_method="hist",
                random_state=SEED + 500 + h,
                n_jobs=-1,
                **best_params_global,
            )
            model_h.fit(X_fit_h.values, y_fit_h)

            fi = pd.DataFrame(
                {
                    "feature": X_fit_h.columns,
                    "importance": model_h.feature_importances_,
                    "horizonte": h,
                }
            ).sort_values("importance", ascending=False)
            importance_frames.append(fi)

        df_importance = pd.concat(importance_frames, ignore_index=True)
        display(df_importance.groupby("horizonte").head(12))

        fig, axes = plt.subplots(2, 2, figsize=(16, 11))
        axes = axes.flatten()

        for ax, h in zip(axes, audit_horizons):
            top_h = (
                df_importance[df_importance["horizonte"] == h]
                .head(12)
                .sort_values("importance", ascending=True)
            )
            ax.barh(top_h["feature"], top_h["importance"], color="#4c78a8", alpha=0.9)
            ax.set_title(f"Top-12 importancias | horizonte t+{h}")
            ax.set_xlabel("feature_importance")

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "auditoria_feature_importance_horizontes_clave.png", bbox_inches="tight")
        plt.show()

        summary_lines = [
            "Resumen interpretativo:",
            f"- El notebook lee {len(used_features)} features efectivas de un CSV con {len(csv_columns)} columnas.",
            f"- La variable con mayor missingness entre las usadas es {df_feature_audit.iloc[0]['feature']} ({df_feature_audit.iloc[0]['missing_pct_csv']:.2f}%).",
            f"- El R2 medio global es {res_global['R2']:.3f} y por regimen es {res_regime['R2']:.3f}.",
            f"- El RMSE pasa de {rmse_global[0]:.3f} en t+1 a {rmse_global[-1]:.3f} en t+24 para el modelo global.",
            "- Si el R2 cae rapido y las importancias se concentran en pocos lags o en meteo_temp_ext, el problema es de informacion disponible, no de cableado del notebook.",
        ]
        print("\n".join(summary_lines))
        


## 12. Qué se hizo para intentar mejorar el modelo

Hasta aquí ya existe un bloque base razonable. A partir de este punto documentamos explícitamente los intentos de mejora incorporados en el notebook.

En concreto se probaron cuatro líneas:

- **Paso 1**: baselines diarios y semanales
- **Paso 2**: modelado del residuo respecto a `t-24h`
- **Paso 3**: memoria estructural más fuerte (`t-48h`, `t-168h`, medias por misma hora)
- **Paso 4**: diferencias térmicas y meteorológicas futuras

Cada una de estas decisiones buscaba atacar una debilidad concreta observada en el diagnóstico.
    


In [ ]:
    def build_same_hour_yesterday_baseline(idx_array, df_all, target, horizon=24):
        target_series = df_all[target]
        out = np.zeros((len(idx_array), horizon), dtype=float)

        for i, origin_ts in enumerate(idx_array):
            for h in range(1, horizon + 1):
                ref_ts = origin_ts + pd.Timedelta(hours=h - 24)
                if ref_ts in target_series.index:
                    out[i, h - 1] = float(target_series.loc[ref_ts])
                else:
                    out[i, h - 1] = np.nan
        return out


    def build_same_hour_mean_baseline(idx_array, df_all, target, horizon=24, days_back=7):
        target_series = df_all[target]
        out = np.zeros((len(idx_array), horizon), dtype=float)

        for i, origin_ts in enumerate(idx_array):
            for h in range(1, horizon + 1):
                target_ts = origin_ts + pd.Timedelta(hours=h)
                refs = []
                for d in range(1, days_back + 1):
                    ref_ts = target_ts - pd.Timedelta(hours=24 * d)
                    if ref_ts in target_series.index:
                        refs.append(float(target_series.loc[ref_ts]))
                out[i, h - 1] = float(np.mean(refs)) if refs else np.nan
        return out


    baseline_train = build_same_hour_yesterday_baseline(data["idx_train"], data["df_all"], TARGET_HOURLY, HORIZON)
    baseline_val = build_same_hour_yesterday_baseline(data["idx_val"], data["df_all"], TARGET_HOURLY, HORIZON)
    baseline_test = build_same_hour_yesterday_baseline(data["idx_test"], data["df_all"], TARGET_HOURLY, HORIZON)

    baseline7_train = build_same_hour_mean_baseline(data["idx_train"], data["df_all"], TARGET_HOURLY, HORIZON, days_back=7)
    baseline7_val = build_same_hour_mean_baseline(data["idx_val"], data["df_all"], TARGET_HOURLY, HORIZON, days_back=7)
    baseline7_test = build_same_hour_mean_baseline(data["idx_test"], data["df_all"], TARGET_HOURLY, HORIZON, days_back=7)

    train_baseline_mask = np.isfinite(baseline_train).all(axis=1)
    val_baseline_mask = np.isfinite(baseline_val).all(axis=1)
    test_baseline_mask = np.isfinite(baseline_test).all(axis=1)

    train_baseline7_mask = np.isfinite(baseline7_train).all(axis=1)
    val_baseline7_mask = np.isfinite(baseline7_val).all(axis=1)
    test_baseline7_mask = np.isfinite(baseline7_test).all(axis=1)

    print(
        "Filas validas baseline t-24h:",
        f"train={int(train_baseline_mask.sum())}, val={int(val_baseline_mask.sum())}, test={int(test_baseline_mask.sum())}"
    )
    print(
        "Filas validas baseline media 7d:",
        f"train={int(train_baseline7_mask.sum())}, val={int(val_baseline7_mask.sum())}, test={int(test_baseline7_mask.sum())}"
    )

    if not test_baseline_mask.all():
        raise ValueError("El baseline t-24h en test contiene NaN. Revisa la continuidad temporal del dataset horario.")
    if not test_baseline7_mask.all():
        raise ValueError("El baseline de media 7d en test contiene NaN. Revisa la historia disponible del dataset horario.")

    res_baseline = evaluate_multistep(Y_test[test_baseline_mask], baseline_test[test_baseline_mask], "Baseline same hour yesterday")
    res_baseline7 = evaluate_multistep(Y_test[test_baseline7_mask], baseline7_test[test_baseline7_mask], "Baseline same hour mean 7d")

    df_baselines = pd.DataFrame(
        [
            {k: v for k, v in res_baseline.items() if k != "rmse_horizonte"},
            {k: v for k, v in res_baseline7.items() if k != "rmse_horizonte"},
        ]
    )
    display(df_baselines.round(4))
    


In [ ]:
    residual_global_models = {}
    pred_residual_cols = []
    residual_rows = []

    Y_train_residual = Y_train[train_baseline_mask]
    Y_val_residual = Y_val[val_baseline_mask]
    baseline_train_residual = baseline_train[train_baseline_mask]
    baseline_val_residual = baseline_val[val_baseline_mask]

    for h in REPRESENTATIVE_HORIZONS:
        X_train_h_res = data["direct"][h]["X_train"].iloc[train_baseline_mask].copy()
        X_val_h_res = data["direct"][h]["X_val"].iloc[val_baseline_mask].copy()
        y_train_res_h = Y_train_residual[:, h - 1] - baseline_train_residual[:, h - 1]
        y_val_res_h = Y_val_residual[:, h - 1] - baseline_val_residual[:, h - 1]

        model_h = xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=300,
            tree_method="hist",
            random_state=SEED + 800 + h,
            n_jobs=-1,
            **best_params_global,
        )
        model_h.fit(X_train_h_res.values, y_train_res_h)
        pred_val_res_h = model_h.predict(X_val_h_res.values)
        pred_val_abs_h = clip_predictions(baseline_val_residual[:, h - 1] + pred_val_res_h)
        rmse_val_abs_h = float(np.sqrt(np.mean((Y_val_residual[:, h - 1] - pred_val_abs_h) ** 2)))
        residual_rows.append({"horizonte": h, "RMSE_val_residual_model": rmse_val_abs_h})

    display(pd.DataFrame(residual_rows))

    for h in range(1, HORIZON + 1):
        X_fit_h = pd.concat(
            [
                data["direct"][h]["X_train"].iloc[train_baseline_mask],
                data["direct"][h]["X_val"].iloc[val_baseline_mask],
            ],
            axis=0,
        )
        y_fit_res_h = np.concatenate(
            [
                Y_train_residual[:, h - 1] - baseline_train_residual[:, h - 1],
                Y_val_residual[:, h - 1] - baseline_val_residual[:, h - 1],
            ]
        )

        model_h = xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=300,
            tree_method="hist",
            random_state=SEED + 900 + h,
            n_jobs=-1,
            **best_params_global,
        )
        model_h.fit(X_fit_h.values, y_fit_res_h)
        residual_global_models[h] = model_h

        pred_res_h = model_h.predict(data["direct"][h]["X_test"].values)
        pred_abs_h = clip_predictions(baseline_test[:, h - 1] + pred_res_h)
        pred_residual_cols.append(pred_abs_h)

    pred_global_residual = np.column_stack(pred_residual_cols)
    res_global_residual = evaluate_multistep(Y_test, pred_global_residual, "XGBoost Direct-24 Residual t-24h")

    df_compare_residual = pd.DataFrame(
        [
            {k: v for k, v in res_baseline.items() if k != "rmse_horizonte"},
            {k: v for k, v in res_baseline7.items() if k != "rmse_horizonte"},
            {k: v for k, v in res_global.items() if k != "rmse_horizonte"},
            {k: v for k, v in res_regime.items() if k != "rmse_horizonte"},
            {k: v for k, v in res_global_residual.items() if k != "rmse_horizonte"},
        ]
    )
    display(df_compare_residual.round(4))
    


In [ ]:
    def build_target_hour_diagnostics(y_true, y_pred, idx_origins, model_name, horizon=24):
        rows = []
        for row_idx, origin_ts in enumerate(idx_origins):
            for h in range(1, horizon + 1):
                target_ts = origin_ts + pd.Timedelta(hours=h)
                rows.append(
                    {
                        "modelo": model_name,
                        "origin_ts": origin_ts,
                        "target_ts": target_ts,
                        "target_hour": int(target_ts.hour),
                        "horizon": h,
                        "y_true": float(y_true[row_idx, h - 1]),
                        "y_pred": float(y_pred[row_idx, h - 1]),
                        "abs_error": float(abs(y_true[row_idx, h - 1] - y_pred[row_idx, h - 1])),
                        "sq_error": float((y_true[row_idx, h - 1] - y_pred[row_idx, h - 1]) ** 2),
                    }
                )
        return pd.DataFrame(rows)


    df_diag_hours = pd.concat(
        [
            build_target_hour_diagnostics(Y_test, baseline_test, data["idx_test"], "Baseline t-24h", HORIZON),
            build_target_hour_diagnostics(Y_test, baseline7_test, data["idx_test"], "Baseline mean 7d", HORIZON),
            build_target_hour_diagnostics(Y_test, pred_global, data["idx_test"], "XGB absoluto", HORIZON),
            build_target_hour_diagnostics(Y_test, pred_global_residual, data["idx_test"], "XGB residual", HORIZON),
        ],
        ignore_index=True,
    )

    df_hour_metrics = (
        df_diag_hours.groupby(["modelo", "target_hour"])
        .agg(
            MAE=("abs_error", "mean"),
            RMSE=("sq_error", lambda s: float(np.sqrt(np.mean(s)))),
            consumo_medio_real=("y_true", "mean"),
        )
        .reset_index()
    )
    display(df_hour_metrics.head(32))

    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

    sns.lineplot(
        data=df_hour_metrics,
        x="target_hour",
        y="RMSE",
        hue="modelo",
        marker="o",
        linewidth=2.2,
        ax=axes[0],
    )
    axes[0].set_title("RMSE por hora objetivo")
    axes[0].set_ylabel("RMSE")

    sns.lineplot(
        data=df_hour_metrics,
        x="target_hour",
        y="MAE",
        hue="modelo",
        marker="o",
        linewidth=2.2,
        ax=axes[1],
    )
    axes[1].set_title("MAE por hora objetivo")
    axes[1].set_xlabel("Hora objetivo")
    axes[1].set_ylabel("MAE")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "diagnostico_error_por_hora_objetivo.png", bbox_inches="tight")
    plt.show()

    peak_hours = (
        df_hour_metrics[df_hour_metrics["modelo"] == "XGB absoluto"]
        .sort_values("RMSE", ascending=False)
        .head(6)
    )
    print("Horas objetivo con peor RMSE en el XGB absoluto:")
    display(peak_hours)
    


In [ ]:
    df_final_extended = pd.DataFrame(
        [
            {k: v for k, v in res_baseline.items() if k != "rmse_horizonte"},
            {k: v for k, v in res_baseline7.items() if k != "rmse_horizonte"},
            {k: v for k, v in res_global.items() if k != "rmse_horizonte"},
            {k: v for k, v in res_regime.items() if k != "rmse_horizonte"},
            {k: v for k, v in res_global_residual.items() if k != "rmse_horizonte"},
        ]
    ).set_index("modelo").sort_values("RMSE")

    print("=== COMPARATIVA EXTENDIDA ===")
    print(df_final_extended.round(4).to_string())

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metric_specs = [("RMSE", "#e45756"), ("R2", "#54a24b"), ("sMAPE", "#f58518")]
    for ax, (metric, color) in zip(axes, metric_specs):
        vals = df_final_extended[metric]
        bars = ax.bar(vals.index, vals.values, color=color, alpha=0.88, edgecolor="white")
        ax.set_title(metric)
        ax.tick_params(axis="x", rotation=20)
        offset = max(np.nanmax(np.abs(vals.values)) * 0.02, 1e-6)
        for bar, val in zip(bars, vals.values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + offset, f"{val:.3f}", ha="center", fontsize=8)

    plt.suptitle("Paso 1 y 2 - comparativa con baselines y modelo residual")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "comparativa_baselines_vs_residual.png", bbox_inches="tight")
    plt.show()

    df_final_extended.to_csv(MODEL_DIR / "comparativa_xgb_direct24_step1_step2.csv")
    df_hour_metrics.to_csv(MODEL_DIR / "diagnostico_error_por_hora_objetivo.csv", index=False)
    


## 13. Reales vs predicciones: lectura visual del desempeño

Las métricas resumen una parte importante de la historia, pero no toda.

Esta sección permite ver:

- si el modelo sigue el perfil medio del día,
- dónde falla en días concretos,
- si comprime demasiado la señal,
- y cómo se comporta según el horizonte.

Es una sección clave para interpretar el desempeño con criterio de negocio y no solo estadístico.
    


In [ ]:
    def build_prediction_long_df(idx_origins, y_true, predictions_dict, horizon=24):
        rows = []
        for row_idx, origin_ts in enumerate(idx_origins):
            for h in range(1, horizon + 1):
                target_ts = origin_ts + pd.Timedelta(hours=h)
                row = {
                    "origin_ts": origin_ts,
                    "target_ts": target_ts,
                    "target_hour": int(target_ts.hour),
                    "horizon": h,
                    "real": float(y_true[row_idx, h - 1]),
                }
                for model_name, y_pred in predictions_dict.items():
                    row[model_name] = float(y_pred[row_idx, h - 1])
                    row[f"err_{model_name}"] = float(y_pred[row_idx, h - 1] - y_true[row_idx, h - 1])
                rows.append(row)
        return pd.DataFrame(rows)


    predictions_dict = {
        "baseline_t24": baseline_test,
        "baseline_mean7d": baseline7_test,
        "xgb_global": pred_global,
        "xgb_regime": pred_regime,
        "xgb_residual": pred_global_residual,
    }

    df_pred_long = build_prediction_long_df(data["idx_test"], Y_test, predictions_dict, HORIZON)
    display(df_pred_long.head())
    


In [ ]:
    df_hour_profile = (
        df_pred_long.groupby("target_hour")[["real", "baseline_t24", "baseline_mean7d", "xgb_global", "xgb_regime", "xgb_residual"]]
        .mean()
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(df_hour_profile["target_hour"], df_hour_profile["real"], color="#111111", linewidth=3, marker="o", label="Real")
    ax.plot(df_hour_profile["target_hour"], df_hour_profile["baseline_t24"], color="#9c6644", linewidth=2, marker="o", label="Baseline t-24h")
    ax.plot(df_hour_profile["target_hour"], df_hour_profile["baseline_mean7d"], color="#b279a2", linewidth=2, marker="o", label="Baseline media 7d")
    ax.plot(df_hour_profile["target_hour"], df_hour_profile["xgb_global"], color="#e45756", linewidth=2, marker="o", label="XGB Global")
    ax.plot(df_hour_profile["target_hour"], df_hour_profile["xgb_regime"], color="#4c78a8", linewidth=2, marker="o", label="XGB Regimen")
    ax.plot(df_hour_profile["target_hour"], df_hour_profile["xgb_residual"], color="#54a24b", linewidth=2, marker="o", label="XGB Residual")
    ax.axvspan(8, 10, color="#f58518", alpha=0.12, label="Ventana critica 8-10")
    ax.set_title("Perfil medio del consumo real vs predicho por hora objetivo")
    ax.set_xlabel("Hora objetivo")
    ax.set_ylabel("Consumo HVAC (kWh)")
    ax.legend(ncol=2)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "reales_vs_predicciones_perfil_medio.png", bbox_inches="tight")
    plt.show()
    


In [ ]:
    n_cases = min(4, len(data["idx_test"]))
    if n_cases == 0:
        raise ValueError("No hay casos en test para graficar.")

    selected_pos = np.linspace(0, len(data["idx_test"]) - 1, n_cases, dtype=int)
    fig, axes = plt.subplots(n_cases, 1, figsize=(15, 4.5 * n_cases), sharex=False)
    if n_cases == 1:
        axes = [axes]

    for ax, pos in zip(axes, selected_pos):
        row = df_pred_long[df_pred_long["origin_ts"] == data["idx_test"][pos]].copy()
        row = row.sort_values("target_ts")

        ax.plot(row["target_ts"], row["real"], color="#111111", linewidth=3, marker="o", label="Real")
        ax.plot(row["target_ts"], row["baseline_t24"], color="#9c6644", linewidth=1.8, marker="o", alpha=0.85, label="Baseline t-24h")
        ax.plot(row["target_ts"], row["baseline_mean7d"], color="#b279a2", linewidth=1.8, marker="o", alpha=0.85, label="Baseline media 7d")
        ax.plot(row["target_ts"], row["xgb_global"], color="#e45756", linewidth=1.8, marker="o", alpha=0.90, label="XGB Global")
        ax.plot(row["target_ts"], row["xgb_regime"], color="#4c78a8", linewidth=1.8, marker="o", alpha=0.90, label="XGB Regimen")
        ax.plot(row["target_ts"], row["xgb_residual"], color="#54a24b", linewidth=1.8, marker="o", alpha=0.90, label="XGB Residual")
        ax.axvspan(
            row["target_ts"].min() + pd.Timedelta(hours=7),
            row["target_ts"].min() + pd.Timedelta(hours=9),
            color="#f58518",
            alpha=0.10,
        )
        ax.set_title(f"Pronostico diario desde origen {data['idx_test'][pos]}")
        ax.set_ylabel("kWh")
        ax.grid(alpha=0.20)

    handles, labels = axes[0].get_legend_handles_labels()
    axes[0].legend(handles, labels, ncol=3, loc="upper left")
    axes[-1].set_xlabel("Hora objetivo")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "reales_vs_predicciones_dias_representativos.png", bbox_inches="tight")
    plt.show()
    


In [ ]:
            fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
            scatter_specs = [
                ("xgb_global", "XGB Global", "#e45756"),
                ("xgb_regime", "XGB Regimen", "#4c78a8"),
                ("xgb_residual", "XGB Residual", "#54a24b"),
            ]

            real_vals = df_pred_long["real"].to_numpy()
            lim_min = float(np.nanmin(real_vals))
            lim_max = float(np.nanmax(real_vals))

            for ax, (col, title, color) in zip(axes, scatter_specs):
                pred_vals = df_pred_long[col].to_numpy()
                ax.scatter(real_vals, pred_vals, s=18, alpha=0.28, color=color, edgecolors="none")
                ax.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--", color="black", linewidth=1.4)
                ax.set_title(title)
                ax.set_xlabel("Real")
                ax.set_ylabel("Prediccion")
                ax.grid(alpha=0.20)

            plt.suptitle("Dispersion real vs prediccion en el conjunto de test")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / "reales_vs_predicciones_scatter_test.png", bbox_inches="tight")
            plt.show()
            


In [ ]:
            horizon_plot = [1, 8, 16, 24]
            fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=False, sharey=False)
            axes = axes.flatten()

            for ax, h in zip(axes, horizon_plot):
                sub = df_pred_long[df_pred_long["horizon"] == h].copy().sort_values("target_ts")
                ax.plot(sub["target_ts"], sub["real"], color="#111111", linewidth=2.6, label="Real")
                ax.plot(sub["target_ts"], sub["xgb_global"], color="#e45756", linewidth=1.8, alpha=0.9, label="XGB Global")
                ax.plot(sub["target_ts"], sub["xgb_residual"], color="#54a24b", linewidth=1.8, alpha=0.9, label="XGB Residual")
                ax.set_title(f"Serie temporal en test para horizonte t+{h}")
                ax.set_ylabel("kWh")
                ax.grid(alpha=0.20)

            handles, labels = axes[0].get_legend_handles_labels()
            axes[0].legend(handles, labels, loc="upper left")
            axes[-1].set_xlabel("Tiempo")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / "reales_vs_predicciones_por_horizonte.png", bbox_inches="tight")
            plt.show()

            df_pred_long.to_csv(MODEL_DIR / "predicciones_test_reales_vs_pred.csv", index=False)
            print("Archivo exportado:")
            print(MODEL_DIR / "predicciones_test_reales_vs_pred.csv")
            


## 14. Experimento neuronal 1: LSTM multi-output

Una vez explorado bastante el techo de XGBoost, probamos una red neuronal secuencial sencilla pero comparable:

- entrada: ventana histórica pasada
- salida: vector completo de 24 horas

El objetivo aquí no era sustituir automáticamente a XGBoost, sino comprobar si un modelo secuencial conseguía capturar dependencias temporales que los árboles estuvieran dejando fuera.
    


In [ ]:
            if torch is None:
                raise RuntimeError(f"PyTorch no esta disponible en este entorno: {TORCH_IMPORT_ERROR}")

            LSTM_SEQ_LEN = 48
            LSTM_BATCH_SIZE = 128
            LSTM_EPOCHS = 30
            LSTM_LR = 1e-3
            LSTM_PATIENCE = 6
            LSTM_TRAIN_STRIDE = 1
            LSTM_EVAL_STRIDE = 24

            LSTM_FEATURES = sorted(set(HISTORICAL_FEATURES + [TARGET_HOURLY]))
            print(f"LSTM features: {len(LSTM_FEATURES)}")
            print(LSTM_FEATURES[:20])
            


In [ ]:
            scaler_lstm_x = RobustScaler()
            scaler_lstm_y = RobustScaler()
            scaler_lstm_x.fit(df_train[LSTM_FEATURES])
            scaler_lstm_y.fit(df_train[[TARGET_HOURLY]])

            def scale_lstm_features(df):
                x_scaled = scaler_lstm_x.transform(df[LSTM_FEATURES])
                y_scaled = scaler_lstm_y.transform(df[[TARGET_HOURLY]]).reshape(-1)
                return x_scaled, y_scaled

            x_train_scaled, y_train_scaled = scale_lstm_features(df_train)
            x_val_scaled, y_val_scaled = scale_lstm_features(df_val)
            x_test_scaled, y_test_scaled = scale_lstm_features(df_test)
            


In [ ]:
            class MultiStepSequenceDataset(Dataset):
                def __init__(self, x_scaled, y_scaled, idx_array, seq_len, horizon, stride):
                    self.x_scaled = np.asarray(x_scaled, dtype=np.float32)
                    self.y_scaled = np.asarray(y_scaled, dtype=np.float32)
                    self.idx_array = pd.Index(idx_array)
                    self.seq_len = seq_len
                    self.horizon = horizon
                    self.positions = list(range(0, len(self.x_scaled) - seq_len - horizon + 1, stride))

                def __len__(self):
                    return len(self.positions)

                def __getitem__(self, idx):
                    pos = self.positions[idx]
                    x = self.x_scaled[pos:pos + self.seq_len]
                    y = self.y_scaled[pos + self.seq_len:pos + self.seq_len + self.horizon]
                    origin_ts = self.idx_array[pos + self.seq_len - 1]
                    return (
                        torch.as_tensor(x, dtype=torch.float32),
                        torch.as_tensor(y, dtype=torch.float32),
                        pos,
                        str(origin_ts),
                    )


            def collate_lstm(batch):
                xs = torch.stack([item[0] for item in batch], dim=0)
                ys = torch.stack([item[1] for item in batch], dim=0)
                positions = [item[2] for item in batch]
                origins = [item[3] for item in batch]
                return xs, ys, positions, origins


            ds_lstm_train = MultiStepSequenceDataset(
                x_train_scaled, y_train_scaled, df_train.index, LSTM_SEQ_LEN, HORIZON, LSTM_TRAIN_STRIDE
            )
            ds_lstm_val = MultiStepSequenceDataset(
                x_val_scaled, y_val_scaled, df_val.index, LSTM_SEQ_LEN, HORIZON, LSTM_EVAL_STRIDE
            )
            ds_lstm_test = MultiStepSequenceDataset(
                x_test_scaled, y_test_scaled, df_test.index, LSTM_SEQ_LEN, HORIZON, LSTM_EVAL_STRIDE
            )

            dl_lstm_train = DataLoader(ds_lstm_train, batch_size=LSTM_BATCH_SIZE, shuffle=True, collate_fn=collate_lstm)
            dl_lstm_val = DataLoader(ds_lstm_val, batch_size=LSTM_BATCH_SIZE, shuffle=False, collate_fn=collate_lstm)
            dl_lstm_test = DataLoader(ds_lstm_test, batch_size=LSTM_BATCH_SIZE, shuffle=False, collate_fn=collate_lstm)

            print(f"Secuencias LSTM train: {len(ds_lstm_train):,}")
            print(f"Secuencias LSTM val:   {len(ds_lstm_val):,}")
            print(f"Secuencias LSTM test:  {len(ds_lstm_test):,}")
            


In [ ]:
            class LSTMMultiOutput(nn.Module):
                def __init__(self, input_size, horizon, hidden_size=128, num_layers=2, dropout=0.2):
                    super().__init__()
                    self.lstm = nn.LSTM(
                        input_size=input_size,
                        hidden_size=hidden_size,
                        num_layers=num_layers,
                        dropout=dropout if num_layers > 1 else 0.0,
                        batch_first=True,
                    )
                    self.dropout = nn.Dropout(dropout)
                    self.fc = nn.Sequential(
                        nn.Linear(hidden_size, 64),
                        nn.ReLU(),
                        nn.Dropout(dropout),
                        nn.Linear(64, horizon),
                    )

                def forward(self, x):
                    out, _ = self.lstm(x)
                    out = self.dropout(out[:, -1, :])
                    return self.fc(out)


            lstm_model = LSTMMultiOutput(len(LSTM_FEATURES), HORIZON).to(DEVICE)
            optimizer_lstm = torch.optim.Adam(lstm_model.parameters(), lr=LSTM_LR)
            scheduler_lstm = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_lstm, patience=3, factor=0.5)
            criterion_lstm = nn.HuberLoss()

            print(lstm_model)
            print(f"Parametros entrenables: {sum(p.numel() for p in lstm_model.parameters() if p.requires_grad):,}")
            


In [ ]:
            def train_epoch_lstm(model, loader, optimizer, criterion, device):
                model.train()
                total_loss = 0.0
                for x, y, _, _ in loader:
                    x = x.to(device)
                    y = y.to(device)
                    optimizer.zero_grad(set_to_none=True)
                    pred = model(x)
                    loss = criterion(pred, y)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    total_loss += loss.item() * len(x)
                return total_loss / max(1, len(loader.dataset))


            def eval_loader_lstm(model, loader, criterion, device):
                model.eval()
                total_loss = 0.0
                preds, targets, origin_list = [], [], []
                with torch.no_grad():
                    for x, y, _, origins in loader:
                        x = x.to(device)
                        y = y.to(device)
                        pred = model(x)
                        total_loss += criterion(pred, y).item() * len(x)
                        preds.append(pred.cpu().numpy())
                        targets.append(y.cpu().numpy())
                        origin_list.extend(origins)
                return (
                    total_loss / max(1, len(loader.dataset)),
                    np.concatenate(preds, axis=0),
                    np.concatenate(targets, axis=0),
                    pd.to_datetime(origin_list),
                )


            best_val_loss = float("inf")
            best_weights = None
            patience_cnt = 0
            hist_train_lstm, hist_val_lstm = [], []

            print(f"Entrenando LSTM multi-output | max epochs={LSTM_EPOCHS} | patience={LSTM_PATIENCE}")
            for epoch in range(1, LSTM_EPOCHS + 1):
                loss_train = train_epoch_lstm(lstm_model, dl_lstm_train, optimizer_lstm, criterion_lstm, DEVICE)
                loss_val, _, _, _ = eval_loader_lstm(lstm_model, dl_lstm_val, criterion_lstm, DEVICE)
                hist_train_lstm.append(loss_train)
                hist_val_lstm.append(loss_val)
                scheduler_lstm.step(loss_val)

                improved = loss_val < best_val_loss - 1e-5
                if improved:
                    best_val_loss = loss_val
                    best_weights = {k: v.detach().cpu().clone() for k, v in lstm_model.state_dict().items()}
                    patience_cnt = 0
                else:
                    patience_cnt += 1

                if epoch == 1 or epoch % 5 == 0 or improved:
                    marca = " <- mejor" if improved else ""
                    print(f"Epoca {epoch:3d} | train loss={loss_train:.6f} | val loss={loss_val:.6f}{marca}")

                if patience_cnt >= LSTM_PATIENCE:
                    print(f"Early stopping en epoca {epoch}")
                    break

            if best_weights is not None:
                lstm_model.load_state_dict(best_weights)
            print(f"Mejor val loss LSTM: {best_val_loss:.6f}")

            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(hist_train_lstm, label="Train loss", color="steelblue")
            ax.plot(hist_val_lstm, label="Val loss", color="crimson")
            ax.set_title("LSTM multi-output - curva de aprendizaje")
            ax.set_xlabel("Epoca")
            ax.set_ylabel("Huber Loss")
            ax.legend()
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / "lstm_multioutput_learning_curve.png", bbox_inches="tight")
            plt.show()
            


In [ ]:
            _, pred_lstm_scaled, y_lstm_test_scaled, idx_lstm_test = eval_loader_lstm(
                lstm_model, dl_lstm_test, criterion_lstm, DEVICE
            )

            pred_lstm = scaler_lstm_y.inverse_transform(pred_lstm_scaled.reshape(-1, 1)).reshape(-1, HORIZON)
            y_lstm_test = scaler_lstm_y.inverse_transform(y_lstm_test_scaled.reshape(-1, 1)).reshape(-1, HORIZON)
            pred_lstm = clip_predictions(pred_lstm)

            res_lstm = evaluate_multistep(y_lstm_test, pred_lstm, "LSTM Direct-24 MultiOutput")
            print(pd.DataFrame([{k: v for k, v in res_lstm.items() if k != "rmse_horizonte"}]).round(4).to_string(index=False))
            


In [ ]:
            df_compare_nn = pd.DataFrame(
                [
                    {k: v for k, v in res_baseline.items() if k != "rmse_horizonte"},
                    {k: v for k, v in res_baseline7.items() if k != "rmse_horizonte"},
                    {k: v for k, v in res_global.items() if k != "rmse_horizonte"},
                    {k: v for k, v in res_regime.items() if k != "rmse_horizonte"},
                    {k: v for k, v in res_global_residual.items() if k != "rmse_horizonte"},
                    {k: v for k, v in res_lstm.items() if k != "rmse_horizonte"},
                ]
            ).set_index("modelo").sort_values("RMSE")

            print("=== COMPARATIVA CON LSTM ===")
            print(df_compare_nn.round(4).to_string())

            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            for ax, (metric, color) in zip(
                axes,
                [("RMSE", "#e45756"), ("R2", "#54a24b"), ("MAE", "#4c78a8")],
            ):
                vals = df_compare_nn[metric]
                bars = ax.bar(vals.index, vals.values, color=color, alpha=0.88, edgecolor="white")
                ax.set_title(metric)
                ax.tick_params(axis="x", rotation=20)
                offset = max(np.nanmax(np.abs(vals.values)) * 0.02, 1e-6)
                for bar, val in zip(bars, vals.values):
                    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + offset, f"{val:.3f}", ha="center", fontsize=8)

            plt.suptitle("Comparativa final con LSTM multi-output")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / "comparativa_final_con_lstm.png", bbox_inches="tight")
            plt.show()

            df_compare_nn.to_csv(MODEL_DIR / "comparativa_final_con_lstm.csv")
            


In [ ]:
            def build_prediction_long_df_from_idx(idx_origins, y_true, predictions_dict, horizon=24):
                rows = []
                for row_idx, origin_ts in enumerate(idx_origins):
                    for h in range(1, horizon + 1):
                        target_ts = origin_ts + pd.Timedelta(hours=h)
                        row = {
                            "origin_ts": origin_ts,
                            "target_ts": target_ts,
                            "target_hour": int(target_ts.hour),
                            "horizon": h,
                            "real": float(y_true[row_idx, h - 1]),
                        }
                        for model_name, y_pred in predictions_dict.items():
                            row[model_name] = float(y_pred[row_idx, h - 1])
                        rows.append(row)
                return pd.DataFrame(rows)


            df_pred_lstm_long = build_prediction_long_df_from_idx(
                idx_lstm_test,
                y_lstm_test,
                {"lstm_direct24": pred_lstm},
                HORIZON,
            )

            fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=False, sharey=False)
            axes = axes.flatten()
            for ax, h in zip(axes, [1, 8, 16, 24]):
                sub = df_pred_lstm_long[df_pred_lstm_long["horizon"] == h].copy().sort_values("target_ts")
                ax.plot(sub["target_ts"], sub["real"], color="#111111", linewidth=2.4, label="Real")
                ax.plot(sub["target_ts"], sub["lstm_direct24"], color="#7b61ff", linewidth=1.8, label="LSTM Direct-24")
                ax.set_title(f"LSTM en test para horizonte t+{h}")
                ax.set_ylabel("kWh")
                ax.grid(alpha=0.20)

            handles, labels = axes[0].get_legend_handles_labels()
            axes[0].legend(handles, labels, loc="upper left")
            axes[-1].set_xlabel("Tiempo")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / "lstm_reales_vs_predicciones_por_horizonte.png", bbox_inches="tight")
            plt.show()
            


## 15. Experimento neuronal 2: LSTM seq2seq con covariables futuras

El experimento LSTM anterior todavía tenía una desventaja frente al enfoque `Direct-24`: solo veía historia pasada.

Por eso aquí probamos una versión más justa:

- un `encoder-decoder LSTM`
- historia pasada en el encoder
- covariables futuras conocidas en el decoder

Este experimento sirve para comprobar si una arquitectura secuencial más alineada con la lógica del problema mejora realmente el desempeño.
    


In [ ]:
            if torch is None:
                raise RuntimeError(f"PyTorch no esta disponible en este entorno: {TORCH_IMPORT_ERROR}")

            SEQ2SEQ_SEQ_LEN = 72
            SEQ2SEQ_BATCH_SIZE = 128
            SEQ2SEQ_EPOCHS = 25
            SEQ2SEQ_LR = 1e-3
            SEQ2SEQ_PATIENCE = 6
            SEQ2SEQ_TRAIN_STRIDE = 1
            SEQ2SEQ_EVAL_STRIDE = 24

            LSTM_HIST_FEATURES = sorted(set(HISTORICAL_FEATURES + [TARGET_HOURLY]))
            LSTM_FUTURE_FEATURES = sorted(set(FUTURE_KNOWN_FEATURES))

            print(f"Hist features seq2seq: {len(LSTM_HIST_FEATURES)}")
            print(f"Future features seq2seq: {len(LSTM_FUTURE_FEATURES)}")
            


In [ ]:
            scaler_seq2seq_hist = RobustScaler()
            scaler_seq2seq_fut = RobustScaler()
            scaler_seq2seq_y = RobustScaler()

            scaler_seq2seq_hist.fit(df_train[LSTM_HIST_FEATURES])
            scaler_seq2seq_fut.fit(df_train[LSTM_FUTURE_FEATURES])
            scaler_seq2seq_y.fit(df_train[[TARGET_HOURLY]])


            class Seq2SeqDirect24Dataset(Dataset):
                def __init__(self, df_split, hist_features, future_features, target, seq_len, horizon, stride):
                    self.df = df_split.copy()
                    self.hist_features = hist_features
                    self.future_features = future_features
                    self.target = target
                    self.seq_len = seq_len
                    self.horizon = horizon
                    self.stride = stride

                    self.hist_scaled = scaler_seq2seq_hist.transform(self.df[self.hist_features]).astype(np.float32)
                    self.future_scaled = scaler_seq2seq_fut.transform(self.df[self.future_features]).astype(np.float32)
                    self.y_scaled = scaler_seq2seq_y.transform(self.df[[self.target]]).reshape(-1).astype(np.float32)
                    self.idx = pd.Index(self.df.index)

                    max_start = len(self.df) - self.seq_len - self.horizon + 1
                    self.positions = []
                    for pos in range(0, max_start, self.stride):
                        full_idx = self.idx[pos:pos + self.seq_len + self.horizon]
                        if len(full_idx) != self.seq_len + self.horizon:
                            continue
                        if full_idx.to_series().diff().iloc[1:].eq(pd.Timedelta(hours=1)).all():
                            self.positions.append(pos)

                def __len__(self):
                    return len(self.positions)

                def __getitem__(self, idx):
                    pos = self.positions[idx]
                    x_hist = self.hist_scaled[pos:pos + self.seq_len]
                    x_future = self.future_scaled[pos + self.seq_len:pos + self.seq_len + self.horizon]
                    y = self.y_scaled[pos + self.seq_len:pos + self.seq_len + self.horizon]
                    origin_ts = self.idx[pos + self.seq_len - 1]
                    return (
                        torch.as_tensor(x_hist, dtype=torch.float32),
                        torch.as_tensor(x_future, dtype=torch.float32),
                        torch.as_tensor(y, dtype=torch.float32),
                        str(origin_ts),
                    )


            def collate_seq2seq(batch):
                x_hist = torch.stack([item[0] for item in batch], dim=0)
                x_future = torch.stack([item[1] for item in batch], dim=0)
                y = torch.stack([item[2] for item in batch], dim=0)
                origins = [item[3] for item in batch]
                return x_hist, x_future, y, origins


            ds_seq2seq_train = Seq2SeqDirect24Dataset(
                df_train, LSTM_HIST_FEATURES, LSTM_FUTURE_FEATURES, TARGET_HOURLY, SEQ2SEQ_SEQ_LEN, HORIZON, SEQ2SEQ_TRAIN_STRIDE
            )
            ds_seq2seq_val = Seq2SeqDirect24Dataset(
                df_val, LSTM_HIST_FEATURES, LSTM_FUTURE_FEATURES, TARGET_HOURLY, SEQ2SEQ_SEQ_LEN, HORIZON, SEQ2SEQ_EVAL_STRIDE
            )
            ds_seq2seq_test = Seq2SeqDirect24Dataset(
                df_test, LSTM_HIST_FEATURES, LSTM_FUTURE_FEATURES, TARGET_HOURLY, SEQ2SEQ_SEQ_LEN, HORIZON, SEQ2SEQ_EVAL_STRIDE
            )

            dl_seq2seq_train = DataLoader(ds_seq2seq_train, batch_size=SEQ2SEQ_BATCH_SIZE, shuffle=True, collate_fn=collate_seq2seq)
            dl_seq2seq_val = DataLoader(ds_seq2seq_val, batch_size=SEQ2SEQ_BATCH_SIZE, shuffle=False, collate_fn=collate_seq2seq)
            dl_seq2seq_test = DataLoader(ds_seq2seq_test, batch_size=SEQ2SEQ_BATCH_SIZE, shuffle=False, collate_fn=collate_seq2seq)

            print(f"Secuencias seq2seq train: {len(ds_seq2seq_train):,}")
            print(f"Secuencias seq2seq val:   {len(ds_seq2seq_val):,}")
            print(f"Secuencias seq2seq test:  {len(ds_seq2seq_test):,}")
            


In [ ]:
            class Seq2SeqLSTM(nn.Module):
                def __init__(self, hist_input_size, future_input_size, horizon, hidden_size=128, num_layers=2, dropout=0.2):
                    super().__init__()
                    self.encoder = nn.LSTM(
                        input_size=hist_input_size,
                        hidden_size=hidden_size,
                        num_layers=num_layers,
                        dropout=dropout if num_layers > 1 else 0.0,
                        batch_first=True,
                    )
                    self.decoder = nn.LSTM(
                        input_size=future_input_size,
                        hidden_size=hidden_size,
                        num_layers=num_layers,
                        dropout=dropout if num_layers > 1 else 0.0,
                        batch_first=True,
                    )
                    self.dropout = nn.Dropout(dropout)
                    self.proj = nn.Sequential(
                        nn.Linear(hidden_size, 64),
                        nn.ReLU(),
                        nn.Dropout(dropout),
                        nn.Linear(64, 1),
                    )
                    self.horizon = horizon

                def forward(self, x_hist, x_future):
                    _, (h, c) = self.encoder(x_hist)
                    dec_out, _ = self.decoder(x_future, (h, c))
                    dec_out = self.dropout(dec_out)
                    pred = self.proj(dec_out).squeeze(-1)
                    return pred


            lstm_seq2seq_model = Seq2SeqLSTM(
                len(LSTM_HIST_FEATURES),
                len(LSTM_FUTURE_FEATURES),
                HORIZON,
            ).to(DEVICE)

            optimizer_seq2seq = torch.optim.Adam(lstm_seq2seq_model.parameters(), lr=SEQ2SEQ_LR)
            scheduler_seq2seq = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_seq2seq, patience=3, factor=0.5)
            criterion_seq2seq = nn.HuberLoss()

            print(lstm_seq2seq_model)
            print(f"Parametros entrenables seq2seq: {sum(p.numel() for p in lstm_seq2seq_model.parameters() if p.requires_grad):,}")
            


In [ ]:
            def train_epoch_seq2seq(model, loader, optimizer, criterion, device):
                model.train()
                total_loss = 0.0
                for x_hist, x_future, y, _ in loader:
                    x_hist = x_hist.to(device)
                    x_future = x_future.to(device)
                    y = y.to(device)
                    optimizer.zero_grad(set_to_none=True)
                    pred = model(x_hist, x_future)
                    loss = criterion(pred, y)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    total_loss += loss.item() * len(x_hist)
                return total_loss / max(1, len(loader.dataset))


            def eval_loader_seq2seq(model, loader, criterion, device):
                model.eval()
                total_loss = 0.0
                preds, targets, origin_list = [], [], []
                with torch.no_grad():
                    for x_hist, x_future, y, origins in loader:
                        x_hist = x_hist.to(device)
                        x_future = x_future.to(device)
                        y = y.to(device)
                        pred = model(x_hist, x_future)
                        total_loss += criterion(pred, y).item() * len(x_hist)
                        preds.append(pred.cpu().numpy())
                        targets.append(y.cpu().numpy())
                        origin_list.extend(origins)
                return (
                    total_loss / max(1, len(loader.dataset)),
                    np.concatenate(preds, axis=0),
                    np.concatenate(targets, axis=0),
                    pd.to_datetime(origin_list),
                )


            best_val_loss_seq2seq = float("inf")
            best_weights_seq2seq = None
            patience_cnt_seq2seq = 0
            hist_train_seq2seq, hist_val_seq2seq = [], []

            print(f"Entrenando LSTM seq2seq | max epochs={SEQ2SEQ_EPOCHS} | patience={SEQ2SEQ_PATIENCE}")
            for epoch in range(1, SEQ2SEQ_EPOCHS + 1):
                loss_train = train_epoch_seq2seq(lstm_seq2seq_model, dl_seq2seq_train, optimizer_seq2seq, criterion_seq2seq, DEVICE)
                loss_val, _, _, _ = eval_loader_seq2seq(lstm_seq2seq_model, dl_seq2seq_val, criterion_seq2seq, DEVICE)
                hist_train_seq2seq.append(loss_train)
                hist_val_seq2seq.append(loss_val)
                scheduler_seq2seq.step(loss_val)

                improved = loss_val < best_val_loss_seq2seq - 1e-5
                if improved:
                    best_val_loss_seq2seq = loss_val
                    best_weights_seq2seq = {k: v.detach().cpu().clone() for k, v in lstm_seq2seq_model.state_dict().items()}
                    patience_cnt_seq2seq = 0
                else:
                    patience_cnt_seq2seq += 1

                if epoch == 1 or epoch % 5 == 0 or improved:
                    marca = " <- mejor" if improved else ""
                    print(f"Epoca {epoch:3d} | train loss={loss_train:.6f} | val loss={loss_val:.6f}{marca}")

                if patience_cnt_seq2seq >= SEQ2SEQ_PATIENCE:
                    print(f"Early stopping seq2seq en epoca {epoch}")
                    break

            if best_weights_seq2seq is not None:
                lstm_seq2seq_model.load_state_dict(best_weights_seq2seq)
            print(f"Mejor val loss LSTM seq2seq: {best_val_loss_seq2seq:.6f}")

            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(hist_train_seq2seq, label="Train loss", color="steelblue")
            ax.plot(hist_val_seq2seq, label="Val loss", color="darkgreen")
            ax.set_title("LSTM seq2seq - curva de aprendizaje")
            ax.set_xlabel("Epoca")
            ax.set_ylabel("Huber Loss")
            ax.legend()
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / "lstm_seq2seq_learning_curve.png", bbox_inches="tight")
            plt.show()
            


In [ ]:
            _, pred_seq2seq_scaled, y_seq2seq_test_scaled, idx_seq2seq_test = eval_loader_seq2seq(
                lstm_seq2seq_model, dl_seq2seq_test, criterion_seq2seq, DEVICE
            )

            pred_seq2seq = scaler_seq2seq_y.inverse_transform(pred_seq2seq_scaled.reshape(-1, 1)).reshape(-1, HORIZON)
            y_seq2seq_test = scaler_seq2seq_y.inverse_transform(y_seq2seq_test_scaled.reshape(-1, 1)).reshape(-1, HORIZON)
            pred_seq2seq = clip_predictions(pred_seq2seq)

            res_lstm_seq2seq = evaluate_multistep(y_seq2seq_test, pred_seq2seq, "LSTM Seq2Seq Direct-24")
            print(pd.DataFrame([{k: v for k, v in res_lstm_seq2seq.items() if k != "rmse_horizonte"}]).round(4).to_string(index=False))
            


In [ ]:
            df_compare_seq2seq = pd.DataFrame(
                [
                    {k: v for k, v in res_baseline.items() if k != "rmse_horizonte"},
                    {k: v for k, v in res_baseline7.items() if k != "rmse_horizonte"},
                    {k: v for k, v in res_global.items() if k != "rmse_horizonte"},
                    {k: v for k, v in res_regime.items() if k != "rmse_horizonte"},
                    {k: v for k, v in res_global_residual.items() if k != "rmse_horizonte"},
                    {k: v for k, v in res_lstm.items() if k != "rmse_horizonte"},
                    {k: v for k, v in res_lstm_seq2seq.items() if k != "rmse_horizonte"},
                ]
            ).set_index("modelo").sort_values("RMSE")

            print("=== COMPARATIVA CON LSTM SEQ2SEQ ===")
            print(df_compare_seq2seq.round(4).to_string())

            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            for ax, (metric, color) in zip(
                axes,
                [("RMSE", "#e45756"), ("R2", "#54a24b"), ("MAE", "#4c78a8")],
            ):
                vals = df_compare_seq2seq[metric]
                bars = ax.bar(vals.index, vals.values, color=color, alpha=0.88, edgecolor="white")
                ax.set_title(metric)
                ax.tick_params(axis="x", rotation=20)
                offset = max(np.nanmax(np.abs(vals.values)) * 0.02, 1e-6)
                for bar, val in zip(bars, vals.values):
                    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + offset, f"{val:.3f}", ha="center", fontsize=8)

            plt.suptitle("Comparativa final con LSTM seq2seq")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / "comparativa_final_con_lstm_seq2seq.png", bbox_inches="tight")
            plt.show()

            df_compare_seq2seq.to_csv(MODEL_DIR / "comparativa_final_con_lstm_seq2seq.csv")
            


## 16. Interpretación de resultados y lectura de negocio

Después de comparar modelos, la pregunta relevante ya no es solo “cuál gana en una métrica”, sino:

- si la mejora es suficiente para apoyar planificación real,
- si el modelo captura bien la estructura diaria,
- y si el error restante es compatible con una toma de decisiones razonable.

En la práctica, este notebook muestra que:

- XGBoost sigue siendo la familia más sólida con el dataset disponible,
- la segmentación por régimen aporta algo de valor,
- las mejoras por memoria temporal ayudan, pero de forma moderada,
- y las redes neuronales no lograron superar a los árboles en esta formulación.
    


In [ ]:
    available_results = []
    for var_name in ["res_baseline", "res_baseline7", "res_global", "res_regime", "res_global_residual", "res_lstm", "res_lstm_seq2seq"]:
        if var_name in globals():
            result = globals()[var_name]
            available_results.append({k: v for k, v in result.items() if k != "rmse_horizonte"})

    df_summary_all = pd.DataFrame(available_results).drop_duplicates(subset="modelo", keep="last")
    if not df_summary_all.empty:
        df_summary_all = df_summary_all.set_index("modelo").sort_values("RMSE")
        print("=== RESUMEN EJECUTIVO DEL BENCHMARK ===")
        print(df_summary_all.round(4).to_string())

        mejor_rmse = df_summary_all["RMSE"].idxmin()
        mejor_mae = df_summary_all["MAE"].idxmin()
        mejor_r2 = df_summary_all["R2"].idxmax()

        print("\nLectura rápida:")
        print(f"- Mejor RMSE: {mejor_rmse}")
        print(f"- Mejor MAE:  {mejor_mae}")
        print(f"- Mejor R2:   {mejor_r2}")
    


## 17. Diagnóstico crítico del desempeño

Esta es la parte más importante del notebook desde el punto de vista metodológico.

El comportamiento observado sugiere lo siguiente:

- el modelo sí captura una parte útil del patrón horario y semanal,
- pero la señal predictiva sigue siendo moderada,
- la franja de arranque del edificio continúa siendo una fuente importante de error,
- y buena parte del límite parece venir de variables no observadas directamente.

En otras palabras: no parece un problema de “falta de complejidad del algoritmo”, sino de techo informativo del dataset disponible.
    


## 18. Mejoras futuras

Si este trabajo continuara, las mejoras más prometedoras no serían solo de tuning. Conviene separarlas en cuatro frentes:

**Datos**
- Incorporar proxies mejores de ocupación real.
- Añadir eventos operativos del edificio y del sistema HVAC.
- Incluir variables de control, setpoints o estados del equipo si están disponibles.

**Features**
- Refinar señales de arranque y transición.
- Explorar variables de diferencia térmica más específicas por estación o por régimen.
- Construir features semanales más ricas solo si aportan señal nueva y no redundante.

**Modelado**
- Mantener XGBoost como referencia principal.
- Explorar modelos híbridos solo si se dispone de mejor información futura.
- Considerar modelos en dos etapas si el objetivo fuera separar “demanda baja” vs “demanda operativa”.

**Evaluación y negocio**
- Seguir evaluando por hora objetivo, no solo con métricas globales.
- Priorizar interpretaciones útiles para planificación diaria, no solo pequeñas mejoras numéricas.
- Definir desde negocio qué nivel de error sería realmente aceptable para usar el modelo en operación.
    
